In [7]:
# ============================================================
# GROUPDNA - WhatsApp Group Personality & Activity Analyzer
# Student: Utkarsha Umesh Khade.
# Roll No.: 45
# Batch: A
# ============================================================

# AI-assisted: Used as a learning aid while preparing this code.

from datetime import datetime, timedelta
import numpy as np


# ============================================================
# FEATURE 1 - CHAT PARSER
# ============================================================

FILE_NAME = "/content/hostel_bois.txt (2).txt"


def parse_chat_file(file_name):

    messages = []
    system_messages = []
    media_count = {}
    deleted_count = {}

    with open(file_name, "r", encoding="utf-8") as file:

        for raw_line in file:

            line = raw_line.rstrip("\n")

            if line.strip() == "":
                continue

            parts = line.split(" - ", 1)

            if len(parts) != 2:
                continue

            timestamp_text = parts[0]
            remaining = parts[1]

            timestamp_parts = timestamp_text.split(", ", 1)

            if len(timestamp_parts) != 2:
                continue

            try:
                message_time = datetime.strptime(
                    timestamp_parts[0] + " " + timestamp_parts[1],
                    "%d/%m/%y %H:%M"
                )

            except ValueError:
                continue

            # System message
            if ":" not in remaining:

                system_messages.append({
                    "time": message_time,
                    "text": remaining
                })

                continue

            sender, message = remaining.split(":", 1)

            sender = sender.strip()
            message = message.strip()

            if sender == "":
                continue

            messages.append({
                "time": message_time,
                "sender": sender,
                "message": message
            })

            # Media
            if message == "<Media omitted>":

                media_count[sender] = media_count.get(sender, 0) + 1

            # Deleted messages
            if message == "This message was deleted":

                deleted_count[sender] = deleted_count.get(sender, 0) + 1

    return messages, system_messages, media_count, deleted_count


messages, system_messages, media_count, deleted_count = \
    parse_chat_file(FILE_NAME)


print("=" * 60)
print("FEATURE 1 - CHAT PARSER")
print("=" * 60)

print("Real messages :", len(messages))
print("System messages :", len(system_messages))
print("Media entries :", sum(media_count.values()))
print("Deleted messages :", sum(deleted_count.values()))

print("First message :", messages[0]["time"])
print("Last message :", messages[-1]["time"])


# ============================================================
# FEATURE 2 - GROUP OVERVIEW
# ============================================================

def count_by_sender(records):

    counts = {}

    for row in records:

        name = row["sender"]

        counts[name] = counts.get(name, 0) + 1

    return counts


message_counts = count_by_sender(messages)

participants = list(message_counts.keys())

start_date = min(row["time"] for row in messages).date()
end_date = max(row["time"] for row in messages).date()

total_messages = len(messages)


print("\n" + "=" * 60)
print("FEATURE 2 - GROUP OVERVIEW")
print("=" * 60)

print("Total messages :", total_messages)
print("Participants :", len(participants))
print(
    "Date range :",
    start_date,
    "to",
    end_date
)

print("\nMESSAGES PER PERSON")

for person in sorted(
    participants,
    key=lambda x: message_counts[x],
    reverse=True
):

    count = message_counts[person]

    percentage = (count / total_messages) * 100

    bar = "█" * max(1, int(percentage / 2))

    print(
        f"{person:<10} "
        f"{bar:<20} "
        f"{count:>4} "
        f"({percentage:.1f}%)"
    )


# ============================================================
# FEATURE 3 - BUSIEST DAY AND HOUR
# ============================================================

def calculate_day_counts(records):

    counts = {}

    for row in records:

        day = row["time"].date()

        counts[day] = counts.get(day, 0) + 1

    return counts


def calculate_hour_counts(records):

    counts = {}

    for row in records:

        hour = row["time"].hour

        counts[hour] = counts.get(hour, 0) + 1

    return counts


daily_counts = calculate_day_counts(messages)
hourly_counts = calculate_hour_counts(messages)

busiest_day = max(
    daily_counts,
    key=daily_counts.get
)

busiest_hour = max(
    hourly_counts,
    key=hourly_counts.get
)


print("\n" + "=" * 60)
print("FEATURE 3 - BUSIEST DAY AND HOUR")
print("=" * 60)

print(
    "Busiest day :",
    busiest_day,
    "(",
    daily_counts[busiest_day],
    "messages )"
)

print(
    "Busiest hour :",
    f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00",
    "(",
    hourly_counts[busiest_hour],
    "messages )"
)


# ============================================================
# FEATURE 4 - NUMPY ACTIVITY HEATMAP
# ============================================================

people = sorted(
    participants,
    key=lambda x: message_counts[x],
    reverse=True
)

# NumPy matrix
activity_matrix = np.zeros(
    (len(people), 24),
    dtype=int
)


for row in messages:

    person_index = people.index(row["sender"])

    hour = row["time"].hour

    activity_matrix[person_index][hour] += 1


def heat_symbol(value, maximum):

    if value == 0:
        return "."

    ratio = value / maximum

    if ratio < 0.25:
        return "░"

    elif ratio < 0.50:
        return "▒"

    elif ratio < 0.75:
        return "▓"

    else:
        return "█"


print("\n" + "=" * 60)
print("FEATURE 4 - NUMPY ACTIVITY HEATMAP")
print("=" * 60)

print("      " + " ".join(
    f"{hour:02d}" for hour in range(24)
))

maximum_value = int(activity_matrix.max())

for index, person in enumerate(people):

    visual = " ".join(
        heat_symbol(
            int(value),
            maximum_value
        )
        for value in activity_matrix[index]
    )

    print(
        f"{person:<8} {visual}"
    )


print("\nNumPy matrix shape :",
      activity_matrix.shape)


# ============================================================
# FEATURE 5 - TOP WORDS
# ============================================================

stop_words = set(
    """
    the a an and or is are was were be been
    to of in on for with from i me my mine
    you your yours he him his she her it its
    we us our they them their this that these
    those am have has had do does did will would
    can could should may might must just very
    more most some any all each every much many
    what when where who whom why how so but if
    then than there here into out up down over
    under again too also only really about because
    while as at by hai hain ho ka ki ke ko mein
    main mera meri mere se par aur ek
    """.split()
)


def clean_word(word):

    punctuation = ". कारक !?;:'\"()[]{}<>—-_\\"

    return word.strip(punctuation).lower()


word_counts = {}


for row in messages:

    message = row["message"]

    if message == "<Media omitted>":
        continue

    if message == "This message was deleted":
        continue

    words = message.split()

    for raw_word in words:

        word = clean_word(raw_word)

        if len(word) > 1 and word not in stop_words:

            word_counts[word] = \
                word_counts.get(word, 0) + 1


top_words = sorted(
    word_counts.items(),
    key=lambda item: (-item[1], item[0])
)[:10]


print("\n" + "=" * 60)
print("FEATURE 5 - TOP WORDS")
print("=" * 60)

for word, count in top_words:

    bar = "█" * min(
        30,
        max(1, count // 5)
    )

    print(
        f"{word:<15} {bar} {count}"
    )


# ============================================================
# FEATURE 6 - RESPONSE TIME
# ============================================================

def calculate_response_gaps(records):

    gaps = {}

    for i in range(1, len(records)):

        previous = records[i - 1]
        current = records[i]

        if previous["sender"] == current["sender"]:
            continue

        difference = (
            current["time"] -
            previous["time"]
        ).total_seconds() / 60

        # Ignore very large gaps
        if 0 <= difference <= 24 * 60:

            person = current["sender"]

            if person not in gaps:
                gaps[person] = []

            gaps[person].append(difference)

    return gaps


response_gaps = calculate_response_gaps(messages)


print("\n" + "=" * 60)
print("FEATURE 6 - RESPONSE PATTERNS")
print("=" * 60)


for person in people:

    values = response_gaps.get(
        person,
        []
    )

    if len(values) > 0:

        average = sum(values) / len(values)

        print(
            f"{person:<10} "
            f"Average reply gap: "
            f"{average:.1f} minutes"
        )

    else:

        print(
            f"{person:<10} "
            "No measurable reply gaps"
        )


# ============================================================
# LONGEST SILENT STREAK
# ============================================================

all_days = []

current_day = start_date

while current_day <= end_date:

    all_days.append(current_day)

    current_day += timedelta(days=1)


def longest_silent_streak(person):

    active_days = set()

    for row in messages:

        if row["sender"] == person:

            active_days.add(
                row["time"].date()
            )

    longest = 0
    current = 0

    longest_start = None
    longest_end = None

    streak_start = None

    for day in all_days:

        if day not in active_days:

            if current == 0:
                streak_start = day

            current += 1

            if current > longest:

                longest = current

                longest_start = streak_start
                longest_end = day

        else:

            current = 0

    return (
        longest,
        longest_start,
        longest_end
    )


silent_results = {}


print("\nLONGEST SILENT STREAKS")

for person in people:

    result = longest_silent_streak(person)

    silent_results[person] = result

    days = result[0]

    if days > 0:

        print(
            f"{person:<10} "
            f"{days} days "
            f"({result[1]} to {result[2]})"
        )

    else:

        print(
            f"{person:<10} 0 days"
        )


# ============================================================
# FEATURE 7 - PERSONALITY ARCHETYPES
# ============================================================

caring_words = {
    "okay",
    "safe",
    "eat",
    "sleep",
    "take",
    "care",
    "please",
    "reminder",
    "drink",
    "water",
    "forget",
    "breakfast",
    "rest"
}


fun_words = {
    "lol",
    "lmao",
    "haha",
    "rofl",
    "lmfao"
}


def average_message_burst(person):

    bursts = []

    current_burst = 0
    previous_sender = None

    for row in messages:

        sender = row["sender"]

        if sender == person:

            if previous_sender == person:

                current_burst += 1

            else:

                if current_burst:
                    bursts.append(current_burst)

                current_burst = 1

            previous_sender = person

        else:

            if previous_sender == person:

                if current_burst:
                    bursts.append(
                        current_burst
                    )

                current_burst = 0

            previous_sender = sender

    if current_burst:
        bursts.append(current_burst)

    if len(bursts) == 0:
        return 0

    return sum(bursts) / len(bursts)


def night_ratio(person):

    own_messages = [
        row for row in messages
        if row["sender"] == person
    ]

    if len(own_messages) == 0:
        return 0

    night_messages = 0

    for row in own_messages:

        hour = row["time"].hour

        if hour >= 23 or hour <= 4:

            night_messages += 1

    return night_messages / len(own_messages)


def average_words(person):

    own_messages = [
        row for row in messages
        if row["sender"] == person
        and row["message"] not in
        ("<Media omitted>",
         "This message was deleted")
    ]

    if len(own_messages) == 0:
        return 0

    total_words = 0

    for row in own_messages:

        total_words += len(
            row["message"].split()
        )

    return total_words / len(own_messages)


def drama_ratio(person):

    own_messages = [
        row for row in messages
        if row["sender"] == person
    ]

    if len(own_messages) == 0:
        return 0

    eligible = 0
    drama = 0

    for row in own_messages:

        text = row["message"]

        letters = ""

        for character in text:

            if character.isalpha():
                letters += character

        if len(letters) >= 3:

            eligible += 1

            if (
                letters.upper() == letters
                and letters.lower() != letters
            ):
                drama += 1

        if text.count("!") >= 2:
            drama += 1

    if eligible == 0:
        return 0

    return drama / eligible


def caring_score(person):

    score = 0

    for row in messages:

        if row["sender"] != person:
            continue

        for raw_word in row["message"].lower().split():

            word = clean_word(raw_word)

            if word in caring_words:

                score += 1

    return score


def comedian_score(person):

    score = 0

    for row in messages:

        if row["sender"] != person:
            continue

        text = row["message"].lower()

        for word in fun_words:

            if word in text:

                score += 1

    return score


def ghost_ratio(person):

    silent_days = silent_results[person][0]

    return silent_days / len(all_days)


def regularity_score(person):

    active_days = set()

    for row in messages:

        if row["sender"] == person:

            active_days.add(
                row["time"].date()
            )

    return len(active_days) / len(all_days)


archetypes = {}
evidence = {}


for person in people:

    scores = {

        "THE SPAMMER":
            average_message_burst(person) / 3,

        "THE GROUP MOM":
            caring_score(person) / 100,

        "THE NIGHT OWL":
            night_ratio(person) / 0.60,

        "THE STORYTELLER":
            average_words(person) / 30,

        "THE DRAMA QUEEN":
            drama_ratio(person) / 0.30,

        "THE GHOST":
            ghost_ratio(person) / 0.60,

        "THE COMEDIAN":
            comedian_score(person) / 5,

        "THE REGULAR":
            regularity_score(person)
    }

    selected = max(
        scores,
        key=scores.get
    )

    archetypes[person] = selected


    if selected == "THE SPAMMER":

        evidence[person] = (
            f"Average burst "
            f"{average_message_burst(person):.1f}"
        )

    elif selected == "THE GROUP MOM":

        evidence[person] = (
            f"Caring score "
            f"{caring_score(person)}"
        )

    elif selected == "THE NIGHT OWL":

        evidence[person] = (
            f"{night_ratio(person)*100:.1f}% "
            "messages at night"
        )

    elif selected == "THE STORYTELLER":

        evidence[person] = (
            f"Average "
            f"{average_words(person):.1f} "
            "words/message"
        )

    elif selected == "THE DRAMA QUEEN":

        evidence[person] = (
            f"{drama_ratio(person)*100:.1f}% "
            "drama/all-caps signal"
        )

    elif selected == "THE GHOST":

        evidence[person] = (
            f"{silent_results[person][0]} "
            "silent days"
        )

    elif selected == "THE COMEDIAN":

        evidence[person] = (
            f"{comedian_score(person)} "
            "humour-keyword hits"
        )

    else:

        evidence[person] = (
            f"Active on "
            f"{regularity_score(person)*100:.1f}% "
            "of days"
        )


print("\n" + "=" * 60)
print("FEATURE 7 - PERSONALITY ARCHETYPES")
print("=" * 60)


for person in people:

    print(
        f"{person:<10} → "
        f"{archetypes[person]:<20} "
        f"({evidence[person]})"
    )


# ============================================================
# FEATURE 8 - FINAL REPORT
# ============================================================

print("\n")
print("=" * 68)
print("       GROUPDNA - FINAL ANALYTICS REPORT")
print("=" * 68)

print(
    "Period       :",
    start_date,
    "to",
    end_date
)

print(
    "Total Messages :",
    total_messages
)

print(
    "Participants   :",
    len(people)
)

print(
    "Busiest Day    :",
    busiest_day,
    "(",
    daily_counts[busiest_day],
    "messages )"
)

print(
    "Busiest Hour   :",
    f"{busiest_hour:02d}:00"
)

print("-" * 68)

print("MESSAGES PER PERSON")

for person in people:

    percentage = (
        message_counts[person]
        / total_messages
        * 100
    )

    print(
        f"{person:<10} "
        f"{message_counts[person]:>4} messages "
        f"({percentage:.1f}%)"
    )


print("-" * 68)

print("TOP 5 WORDS")

for word, count in top_words[:5]:

    print(
        f"{word:<15} {count}"
    )


print("-" * 68)

print("PERSONALITY ARCHETYPES")

for person in people:

    print(
        f"{person:<10} → "
        f"{archetypes[person]:<20} "
        f"{evidence[person]}"
    )


print("-" * 68)

print("DATA QUALITY")

print(
    "Parsed messages :",
    len(messages)
)

print(
    "System messages :",
    len(system_messages)
)

print(
    "Media entries   :",
    sum(media_count.values())
)

print(
    "Deleted messages:",
    sum(deleted_count.values())
)

print("=" * 68)

print(
    "Generated by Utkarsha"
)

print(
    "Built using Python + NumPy"
)

print("=" * 68)

FEATURE 1 - CHAT PARSER
Real messages : 3174
System messages : 4
Media entries : 32
Deleted messages : 15
First message : 2024-04-01 01:17:00
Last message : 2024-05-30 23:31:00

FEATURE 2 - GROUP OVERVIEW
Total messages : 3174
Participants : 6
Date range : 2024-04-01 to 2024-05-30

MESSAGES PER PERSON
Rahul      ███████████████       953 (30.0%)
Priya      ███████████           718 (22.6%)
Neha       ██████████            635 (20.0%)
Aman       ███████               490 (15.4%)
Karan      █████                 354 (11.2%)
Vikas      █                      24 (0.8%)

FEATURE 3 - BUSIEST DAY AND HOUR
Busiest day : 2024-05-04 ( 76 messages )
Busiest hour : 18:00 - 19:00 ( 248 messages )

FEATURE 4 - NUMPY ACTIVITY HEATMAP
      00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul    ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ▓ ▒ ▒ ▓ ▓ ▒ █ ▓ ▒ █ ▓ ▓
Priya    . . . . . . ░ ░ ▒ ▓ ▓ ▓ ▓ ▒ ▒ ▒ ▒ ▒ ▒ ▓ ▒ ▒ ░ ░
Neha     . . . . . ░ ░ ░ ▒ ▒ ▒ ░ ▒ ▒ ▒ ░ ▒ ▒ ▓ ▒ ▒ ▒ ▒ ▒
Aman     ▓ ▓ ▓ ▓ █ . .